<a href="https://colab.research.google.com/github/wGodyn/effective_python/blob/master/lab04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lab 04

## Zadanie 1 (Igor Sieradzki)

Funkcja <t>flow_rate</i> do modyfikacji

In [7]:
def flow_rate(weight, time, *, period=60, units_per_kg=1000):
    """ Funkcja wylicza ile wagi produktu przybyło/ubyło w jednostce czasu """
    return weight / time * period * units_per_kg

weight = 0.5
time = 3

In [8]:
flow = flow_rate(weight, time)
print("{0:.3} kg per second".format(flow))

1e+04 kg per second


Zmodyfikuj funckje <i>flow_rate</i> tak, aby poniższe wywołania działały poprawnie.

In [9]:
flow = flow_rate(weight, time, period=60, units_per_kg=1000)
print("{} grams per minute".format(flow))

10000.0 grams per minute


In [10]:
flow = flow_rate(weight, time, period=1, units_per_kg=1)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [11]:
flow = flow_rate(weight, time)
print("{0:.3} grams per minute".format(flow))

1e+04 grams per minute


Wyjaśnić czemu poniższe rzucanie błędu jest porządane

Rzucenie błędu `TypeError` jest pożądane, ponieważ parametry `period` i `units_per_kg` można przekazywać tylko jako argumenty nazwane. Dzięki temu wywołanie funkcji jest czytelniejsze i wiadomo, co oznaczają podane wartości, np. `period=3600` oraz `units_per_kg=2.2`.

In [12]:
try:
    flow = flow_rate(weight, time, 3600, 2.2)
except TypeError:
    print(True)

True


## Zadanie 2 (Igor Sieradzki)

Dopisz definicje dekoratora _timeit_, który wypisze na wyjscie standardowe czas wywołania udekorowanej funckji np.

```
@timeit
def foo(x):
    return x**2
    
r = foo(2)
```
wypize: <br>
    `Function foo took: 0.00001 seconds` <br>
oraz zwróci 4

In [13]:
from time import time

def timeit(func):
    """ Wypisuje czas wywołania udekorowanej funckji """
    def wrapper(*args, **kwargs):
        start = time()
        result = func(*args, **kwargs)
        end = time()
        print("Function {} took: {} seconds".format(func.__name__, end - start))
        return result
    return wrapper

@timeit
def squares_list(n):
    squares = []
    for i in range(n):
        squares.append(i ** 2)
    return squares

@timeit
def squares_comprehension(n):
    return [i ** 2 for i in range(n)]

@timeit
def squares_map(n):
    return map(lambda x: x**2, range(n))

n = 1000000
l = squares_list(n)
c = squares_comprehension(n)
m = squares_map(n)

Function squares_list took: 0.11166596412658691 seconds
Function squares_comprehension took: 0.09733104705810547 seconds
Function squares_map took: 3.5762786865234375e-06 seconds


## Zadanie 3 (Igor Sieradzki)

Dopisz definicje dekoratora _derivate_ wg. instrukcji w _docstringu_

In [14]:
import sys
sys.float_info.epsilon # epsilon maszynowy


def derivate(epsilon=None):
     """
    Zwraca pochodną funkcji w punkcie, wg. wzoru f'(x) = [f(x+h) - f(x)]/h,
    gdzie h jest parametrem dekoratora, jeśli nie zostanie podany, należy przyjąć 1000 * epsilon maszynowy
    """
     if epsilon is None:
         epsilon = 1000 * sys.float_info.epsilon

     def decorator(func):
         def wrapper(x):
             return (func(x + epsilon) - func(x)) / epsilon
         return wrapper

     return decorator


@derivate(0.01)
def f(x):
    return x*x

@derivate(0.00001)
def g(x):
    return x*x*x+3

def test(a, b, eps=1):
    return abs(round(a)-round(b)) < eps

print(test(f(100), 200.0))
print(round(f(0)) == 0.0)

print(test(g(100), 30000.0))
print(round(g(0)) == 0.0)

from random import random
for x in [random()*1000. for _ in range(20)]:
    print(f(x), 2*x, '\t', test(f(x), 2*x))
    print(g(x), 3*x**2, '\t', test(g(x), 3*x**2))

True
True
True
True
1121.6727634775452 1121.662763476873 	 True
943595.5315828322 943595.5162279315 	 True
504.58605363965034 504.5760536397297 	 True
190947.75319099423 190947.74542998255 	 True
536.1320617535966 536.1220617542577 	 True
215570.15627622601 215570.1488247271 	 True
354.81073253249633 354.8007325329208 	 True
94412.67522051929 94412.6698544229 	 True
1454.4964181724936 1454.4864181729472 	 True
1586648.0767726896 1586648.0554871769 	 True
355.8717155687191 355.86171556881885 	 True
94978.17587107419 94978.1704556872 	 True
900.2267689589644 900.2167689617997 	 True
607792.6859259605 607792.6733400167 	 True
1895.078557531815 1895.0685575418474 	 True
2693463.659286499 2693463.6283378038 	 True
220.44758693118638 220.43758693106463 	 True
36444.55063622445 36444.547298993006 	 True
637.0013356485288 636.9913356496402 	 True
304318.47982108593 304318.47126953444 	 True
1256.6911953967065 1256.6811953966496 	 True
1184435.7371330261 1184435.7201476642 	 True
372.1781398817

## Zadanie 4 (Igor Sieradzki)

Dopisz definicje dekoratora _accepts_ wg. instrukcji w _docstringu_

In [15]:
from inspect import signature

def accepts(*types):
    """Sprawdza czy udekorowanej funckji zostały podane odpowiednie parametry zdefiniowane
       w argumentach dekoratora"""
    def decorator(func):
        def wrapper(*args, **kwargs):
            arguments = signature(func).bind(*args, **kwargs)
            values = arguments.arguments.values()

            for value, expected_type in zip(values, types):
                if not isinstance(value, expected_type):
                    raise TypeError

            return func(*args, **kwargs)
        return wrapper
    return decorator


@accepts(str)
def capitalize(word):
    return word[0].upper() + word[1:]

print(capitalize('ola') == 'Ola')

try:
    capitalize(2)
except TypeError:
    print(True)

@accepts(float, int)
def static_pow(base, exp):
    return base ** exp

print(static_pow(2., 2) == 4.)
print(static_pow(2., exp=2) == 4.)
print(static_pow(base=2., exp=2) == 4.)

try:
    static_pow('x', 10)
except TypeError:
    print(True)

try:
    static_pow(2, 2.2)
except TypeError:
    print(True)

True
True
True
True
True
True
True


## Zadanie 5 (Igor Sieradzki)

Dopisz definicje dekoratora _returns_ wg. instrukcji w _docstringu_

In [16]:
def returns(*types):
    """Sprawdza czy udekorowana funkcja zwraca poprawne argumenty, zdefiniowane w parametrach dekoratora"""
    def decorator(func):
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)

            if len(types) == 1:
                results = (result,)
            else:
                results = result

            if not isinstance(results, tuple) or len(results) != len(types):
                raise TypeError

            for value, expected_type in zip(results, types):
                if not isinstance(value, expected_type):
                    raise TypeError

            return result
        return wrapper
    return decorator


@returns(str)
def str_only_identity(word):
    return word

print(str_only_identity('hello') == 'hello')

try:
    str_only_identity(10)
except TypeError:
    print(True)

@returns(int, int)
def split_indices(x):
    return x[0], x[1]

split = split_indices

print(split_indices(x=[6,9]) == (6,9))

try:
    split('AB')
except TypeError:
    print(True)

True
True
True
True


## Zadanie 6 (Igor Sieradzki)
Stwórz dekorator cached służący do cachowania wywołań dowolnej funkcji, tzn. chcemy by:
* wywołanie funkcji z określonymi argumentami miało miejsce tylko raz
* funkcja mogła przyjmować dowolną liczbę nazwanych i nienazwanych argumentów
* nie musi reagować poprawnie na domyślne argumenty, tzn. wywołanie funkcji z domyślnymi argumentami a podanie dokładnie takich samych może być traktowane jako dwa różne wywołania
* na opakowanej funkcji można wywołać `.cache_reset()`, który usunie cache z pamięci
* wywołanie `.cache_status()` zwraca string z opisem w postaci: <br>
    `Function FUNCTION_NAME called X times, evaluated Y times`

In [17]:
from random import random

def cached(func):
    cache = {}
    called = 0
    evaluated = 0

    def wrapper(*args, **kwargs):
        nonlocal called, evaluated
        called += 1
        key = (args, tuple(sorted(kwargs.items())))

        if key not in cache:
            cache[key] = func(*args, **kwargs)
            evaluated += 1

        return cache[key]

    def cache_reset():
        cache.clear()

    def cache_status():
        return "Function {} called {} times, evaluated {} times".format(
            func.__name__, called, evaluated
        )

    wrapper.cache_reset = cache_reset
    wrapper.cache_status = cache_status
    return wrapper

@cached
def foo(x, y=1, z=4):
    return random()

print(foo(3) == foo(3))
print(foo(4) == foo(4))
print(foo(3, z=-1, y=3) == foo(3, y=3, z=-1))
print(foo(3) != foo(x=3))
a = foo(3)
foo.cache_reset()
print(a != foo(3))
print(foo.cache_status() == 'Function foo called 10 times, evaluated 5 times')

True
True
True
True
True
True


## Zadanie 7 (Krzysztof Hajto)

Napisz dekorator który będzie robić n-krotne złożenie funkcji, gdzie n jest parametrem dekoratora

In [18]:
def zlozenie(n):
    def decorator(func):
        def wrapper(x):
            result = x
            for i in range(n):
                result = func(result)
            return result
        return wrapper
    return decorator

@zlozenie(3)
def f1(x):
    return x+1

@zlozenie(2)
def f2(x):
    return x*x

@zlozenie(5)
def f3(word):
    return "".join(chr(ord(l)+1) for l in word)

print(f1(2)==5)
print(f2(3)==81)
print(f3("alamakota")=="fqfrfptyf")

True
True
True


## Zadanie 8 (Krzysztof Hajto)

Python nie ma wbudowanej instrukcji switch. Ale posiada anonimowe funkcje oraz słowniki. Zaimplementuj poniższy switch w postaci słownika funkcji.
`
int my_function(x, y) {


    switch(x) {
        case 1: return y*y;
        case 2: return x+y;
        case 3: return x*y;
        case 4: return 0;
    }
}
`

PS. Nigdy nie róbcie tego w faktycznym kodzie :)

In [19]:
def my_function(x, y):
    cases = {
        1: lambda: y * y,
        2: lambda: x + y,
        3: lambda: x * y,
        4: lambda: 0
    }
    return cases[x]()

print(my_function(1,3)==9)
print(my_function(2,4)==6)
print(my_function(3,1)==3)
print(my_function(4,9)==0)

True
True
True
True
